In [1]:
from pathlib import Path
from zipfile import ZipFile

ZIP_PATH = Path('/content/BDD-main.zip')
REPO_DIR = Path('/content/BDD-main')
if not (REPO_DIR / 'baseline_pca_ridge.py').exists():
    if not ZIP_PATH.exists():
        from google.colab import files
        uploaded = files.upload()  # 选择 BDD-main.zip
        archives = [name for name in uploaded if name.lower().endswith('.zip')]
        if len(archives) != 1:
            raise ValueError('请上传恰好一个 BDD-main.zip')
        ZIP_PATH.write_bytes(uploaded[archives[0]])

    with ZipFile(ZIP_PATH) as archive:
        root = Path('/content').resolve()
        for member in archive.infolist():
            target = (root / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f'ZIP 中有不安全的路径: {member.filename}')
        archive.extractall(root)

required = [
    'baseline_pca_ridge.py', 'baseline_ridge.py', 'split_data.py',
    'splits/all_cell_types_80_20_seed42.json',
    'splits/all_cell_types_80_20_seed42_train.parquet',
    'splits/all_cell_types_80_20_seed42_validation.parquet',
]
missing = [name for name in required if not (REPO_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'仓库 ZIP 缺少文件: {missing}')
print('仓库文件已就绪:', REPO_DIR)

ModuleNotFoundError: No module named 'google.colab'

Sparse-cell four-fold validation

In [2]:
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(REPO_DIR))
from split_data import get_gene_columns
from baseline_pca_ridge import fit_predict as fit_pca_ridge, mrrmse
from baseline_ridge import fit_predict as fit_direct_ridge

stem = REPO_DIR / 'splits' / 'all_cell_types_80_20_seed42'
original_train = pd.read_parquet(stem.with_name(stem.name + '_train.parquet'))
original_validation = pd.read_parquet(stem.with_name(stem.name + '_validation.parquet'))
manifest = json.loads(stem.with_suffix('.json').read_text(encoding='utf-8'))
pair_cols = ['cell_type', 'sm_name']
for frame, key in [(original_train, 'train_pairs'),
                   (original_validation, 'validation_pairs')]:
    assert list(frame[pair_cols].itertuples(index=False, name=None)) == list(
        map(tuple, manifest[key])
    )
assert original_train.columns.tolist() == original_validation.columns.tolist()
data = pd.concat([original_train, original_validation], ignore_index=True)
assert not data.duplicated(pair_cols).any()
assert len(data) == 614
genes = get_gene_columns(data)
assert len(genes) == 18211
assert pd.api.types.is_bool_dtype(data['control'])
print('已知组合:', len(data), '| 基因:', len(genes))
display(pd.crosstab(data['cell_type'], data['control']))

FileNotFoundError: [Errno 2] No such file or directory: '\\content\\BDD-main\\splits\\all_cell_types_80_20_seed42_train.parquet'

In [3]:
B = 'B cells'
M = 'Myeloid cells'
source_cells = ['NK cells', 'T cells CD4+', 'T cells CD8+',
                'T regulatory cells']
anchors_b = set(data.loc[data.cell_type.eq(B) & ~data.control, 'sm_name'])
anchors_m = set(data.loc[data.cell_type.eq(M) & ~data.control, 'sm_name'])
assert anchors_b == anchors_m and len(anchors_b) == 15
anchor_drugs = anchors_b

folds = []
for target_cell in source_cells:
    is_target = data['cell_type'].eq(target_cell)
    is_held_out = is_target & ~data['control'] & ~data['sm_name'].isin(anchor_drugs)
    train_fold = data.loc[~is_held_out].reset_index(drop=True)
    val_fold = data.loc[is_held_out].reset_index(drop=True)
    anchors_in_target = data.loc[is_target & ~data.control &
                                 data.sm_name.isin(anchor_drugs), 'sm_name']
    assert len(anchors_in_target) in (13, 15)
    assert len(val_fold) > 100 and not val_fold['control'].any()
    assert set(val_fold.sm_name).issubset(set(train_fold.sm_name))
    assert len(train_fold) + len(val_fold) == len(data)
    assert len(train_fold.loc[train_fold.cell_type.eq(target_cell) &
                              ~train_fold.control]) == len(anchors_in_target)
    folds.append((target_cell, train_fold, val_fold))

display(pd.DataFrame([
    {'伪目标细胞': cell, '训练行': len(tr), '该细胞已知常规药物':
     int((tr.cell_type.eq(cell) & ~tr.control).sum()), '验证行': len(va)}
    for cell, tr, va in folds
]))

NameError: name 'data' is not defined

In [4]:
from time import perf_counter

results = {}
fold_results = {}

def evaluate_method(name, predict_function):
    rows = []
    for target_cell, train_fold, val_fold in folds:
        start = perf_counter()
        prediction = predict_function(
            train_fold[pair_cols], train_fold[genes], val_fold[pair_cols]
        )
        assert np.asarray(prediction).shape == (len(val_fold), len(genes))
        assert np.isfinite(np.asarray(prediction)).all()
        rows.append({
            'cell_type': target_cell,
            'MRRMSE': mrrmse(val_fold[genes], prediction),
            'seconds': perf_counter() - start,
        })
    fold_results[name] = pd.DataFrame(rows)
    results[name] = float(fold_results[name]['MRRMSE'].mean())
    display(fold_results[name])
    print(f'{name}: four-fold mean MRRMSE = {results[name]:.6f}')

In [5]:
# Original Ridge settings are unchanged.
evaluate_method('BDD PCA(160) + Ridge',
    lambda x, y, q: fit_pca_ridge(x, y, q, n_components=160, alpha=3.5))
evaluate_method('BDD Ridge (direct)',
    lambda x, y, q: fit_direct_ridge(x, y, q, alpha=4.5))

NameError: name 'folds' is not defined

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

USE_GPU = False
if USE_GPU:
    import subprocess
    subprocess.run(['nvidia-smi', '-L'], check=True)
device = 'cuda' if USE_GPU else 'cpu'

def predict_xgboost(X_train, y_train, X_val, use_pca):
    # A new encoder, PCA and model are fitted for every fold.
    try:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True,
                                dtype=np.float32)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse=True,
                                dtype=np.float32)
    a = encoder.fit_transform(X_train.astype(str))
    b = encoder.transform(X_val.astype(str))
    targets = y_train.to_numpy(dtype=float)
    if use_pca:
        pca = PCA(n_components=160, svd_solver='randomized', random_state=42)
        targets = pca.fit_transform(targets)
    options = {} if use_pca else {'max_bin': 4}
    model = XGBRegressor(
        n_estimators=120, max_depth=3, learning_rate=0.06,
        subsample=0.9, colsample_bytree=0.9,
        objective='reg:squarederror', tree_method='hist', device=device,
        multi_strategy='one_output_per_tree' if use_pca else 'multi_output_tree',
        n_jobs=2, random_state=42, **options,
    )
    model.fit(a, targets.astype(np.float32))
    prediction = model.predict(b)
    return pca.inverse_transform(prediction) if use_pca else prediction

evaluate_method('XGBoost + PCA(160)',
                lambda x, y, q: predict_xgboost(x, y, q, use_pca=True))

In [ ]:
evaluate_method('XGBoost (direct, no PCA)',
                lambda x, y, q: predict_xgboost(x, y, q, use_pca=False))

Four-fold summary

In [ ]:
display(pd.Series(results, name='four_fold_mean_MRRMSE')
        .rename_axis('model').sort_values().to_frame())

Official test predictions: refit on all 614 labeled pairs

In [ ]:
train_df = data.copy()  # Full training data, not one validation fold.

In [ ]:
ID_PATH = Path('/content/id_map.csv')
SAMPLE_PATH = Path('/content/sample_submission.csv')
missing_csv = [p.name for p in (ID_PATH, SAMPLE_PATH) if not p.exists()]
if missing_csv:
    from google.colab import files
    uploaded = files.upload()  # 同时选 id_map.csv 和 sample_submission.csv
    for name in missing_csv:
        if name not in uploaded:
            raise FileNotFoundError(f'未收到 {name}')
        (Path('/content') / name).write_bytes(uploaded[name])

id_map = pd.read_csv(ID_PATH)
sample = pd.read_csv(SAMPLE_PATH)
assert {'id', 'cell_type', 'sm_name'}.issubset(id_map.columns)
assert 'id' in sample.columns and sample.columns[0] == 'id'
assert id_map['id'].is_unique and sample['id'].is_unique
assert set(id_map['id']) == set(sample['id'])
gene_order = sample.columns.drop('id').tolist()
assert len(gene_order) == len(genes) == 18211
assert set(gene_order) == set(genes), 'sample 与训练集基因名不一致'

# reindex 按 sample_submission.csv 的 id 依次查找，不依赖 id_map 的文件行顺序。
test_pairs = id_map.set_index('id').loc[sample['id'], pair_cols].reset_index(drop=True)
assert test_pairs.notna().all().all()
assert test_pairs['cell_type'].isin(['B cells', 'Myeloid cells']).all()
assert not test_pairs.duplicated().any()
known_pairs = set(train_df[pair_cols].itertuples(index=False, name=None))
assert not any(pair in known_pairs for pair in test_pairs.itertuples(index=False, name=None))
assert set(test_pairs['sm_name']).issubset(set(train_df['sm_name']))
print('需要预测的测试组合:', len(test_pairs))
print(test_pairs.groupby('cell_type').size())
print('sample 列数:', len(sample.columns))

In [ ]:
from time import perf_counter
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

X_train = train_df[pair_cols]
y_train = train_df[gene_order]

started = perf_counter()
pred_ridge = fit_pca_ridge(
    X_train, y_train, test_pairs, n_components=160, alpha=3.5,
)
print('Ridge 训练+预测秒数:', round(perf_counter()-started, 1))

started = perf_counter()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True,
                        dtype=np.float32)
a = encoder.fit_transform(X_train.astype(str))
b = encoder.transform(test_pairs.astype(str))
pca = PCA(n_components=160, svd_solver='randomized', random_state=42)
z = pca.fit_transform(y_train.to_numpy(dtype=float))
xgb = XGBRegressor(
    n_estimators=120, max_depth=3, learning_rate=0.06,
    subsample=0.9, colsample_bytree=0.9,
    objective='reg:squarederror', tree_method='hist', device='cpu',
    multi_strategy='one_output_per_tree', n_jobs=2, random_state=42,
)
xgb.fit(a, z.astype(np.float32))
pred_xgb = pca.inverse_transform(xgb.predict(b))
print('XGBoost 训练+预测秒数:', round(perf_counter()-started, 1))

In [ ]:
def write_submission(prediction, filename):
    values = np.asarray(prediction, dtype=float)
    assert values.shape == (len(sample), len(gene_order))
    assert np.isfinite(values).all()
    result = pd.DataFrame(values, columns=gene_order)
    result.insert(0, 'id', sample['id'].to_numpy())
    assert result.columns.tolist() == sample.columns.tolist()
    assert result['id'].equals(sample['id'])
    path = Path('/content') / filename
    result.to_csv(path, index=False, float_format='%.10g')
    print(filename, '| shape:', result.shape,
          '| size MB:', round(path.stat().st_size/1e6, 1))
    return path

ridge_path = write_submission(pred_ridge, 'bdd_ridge_pca_submission.csv')
xgb_path = write_submission(pred_xgb, 'bdd_xgboost_pca_submission.csv')

# 在 Colab 要下载哪个 CSV，就执行对应的 files.download：
from google.colab import files
files.download(xgb_path)
files.download(ridge_path)

In [ ]:
files.download(xgb_path)

In [ ]:
from time import perf_counter
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

encoder = OneHotEncoder(handle_unknown='ignore',sparse_output=True,
                        dtype=np.float32)
a = encoder.fit_transform(train_df[pair_cols].astype(str))
b = encoder.transform(test_pairs.astype(str))
y = train_df[gene_order].to_numpy(dtype=float)

started = perf_counter()
ridge = Ridge(alpha=3.5).fit(a,y)
fitted = ridge.predict(a)
predicted = ridge.predict(b)
center = fitted.mean(axis=0)
svd = PCA(n_components=32,svd_solver='randomized',random_state=42).fit(fitted)
pred_ridge_svd = center + (predicted-center) @ svd.components_.T @ svd.components_
print('Ridge + supervised SVD 秒数:',round(perf_counter()-started,1))

started = perf_counter()
xgb_direct = XGBRegressor(
    n_estimators=120,max_depth=3,learning_rate=0.06,
    subsample=0.9,colsample_bytree=0.9,
    objective='reg:squarederror',tree_method='hist',max_bin=4,
    multi_strategy='multi_output_tree',device='cpu',
    n_jobs=2,random_state=42,
)
xgb_direct.fit(a,y.astype(np.float32))
pred_xgb_direct = xgb_direct.predict(b)
print('XGBoost no-PCA 秒数:',round(perf_counter()-started,1))

In [ ]:
def write_submission(values,name):
    arr = np.asarray(values,dtype=float)
    assert arr.shape == (len(sample),len(gene_order))
    assert np.isfinite(arr).all()
    result = pd.DataFrame(arr,columns=gene_order)
    result.insert(0,'id',sample.id.to_numpy())
    assert result.columns.tolist() == sample.columns.tolist()
    assert result.id.equals(sample.id)
    path = Path('/content')/name
    result.to_csv(path,index=False,float_format='%.8g')
    print(name,'shape',result.shape,'MB',round(path.stat().st_size/1e6,1))
    return path

ridge_path = write_submission(
    pred_ridge_svd,'bdd_ridge_supervised_svd32_alpha3p5_submission.csv')
xgb_path = write_submission(
    pred_xgb_direct,'bdd_xgboost_no_pca_submission.csv')

# 按需下载到电脑：
from google.colab import files
files.download(xgb_path)
files.download(ridge_path)

In [ ]:
from google.colab import files
files.download(xgb_path)
files.download(ridge_path)